In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import random

def create_random_function(n):
    total = 2 ** n

    function_type = random.choice(["constant", "balanced"])

    if function_type == "constant":
        value = random.choice([0, 1])
        function = [value] * total

    else:
        function = [0] * (total // 2) + [1] * (total // 2)
        random.shuffle(function)

    return function, function_type


def create_oracle(n, function):
    qc = QuantumCircuit(n + 1)

    if all(value == 1 for value in function):
        qc.x(n)

    elif not all(value == 0 for value in function):

        for x in range(2 ** n):

            if function[x] == 1:

                binary = format(x, f'0{n}b')
                for i, bit in enumerate(binary):
                    if bit == '0':
                        qc.x(i)

                qc.mcx(list(range(n)), n)
                for i, bit in enumerate(binary):
                    if bit == '0':
                        qc.x(i)

    return qc


def deutsch_jozsa_random(n):

    function, expected_type = create_random_function(n)

    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    oracle = create_oracle(n, function)
    qc.compose(oracle, inplace=True)

    qc.h(range(n))

    qc.measure(range(n), range(n))

    return qc, function, expected_type


n = 3

qc, function, expected_type = deutsch_jozsa_random(n)

print("Generated function:", function)
print("Expected type:", expected_type)
print(qc.draw())

simulator = AerSimulator()

result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print("Measurement:", counts)

if list(counts.keys())[0] == "0" * n:
    predicted = "constant"
else:
    predicted = "balanced"

print("Predicted type:", predicted)

Generated function: [0, 0, 1, 1, 0, 0, 1, 1]
Expected type: balanced
     ┌───┐┌───┐     ┌───┐┌───┐     ┌───┐               ┌───┐┌─┐      
q_0: ┤ H ├┤ X ├──■──┤ X ├┤ X ├──■──┤ X ├──■─────────■──┤ H ├┤M├──────
     ├───┤└───┘  │  └───┘└───┘  │  └───┘  │         │  ├───┤└╥┘┌─┐   
q_1: ┤ H ├───────■──────────────■─────────■─────────■──┤ H ├─╫─┤M├───
     ├───┤┌───┐  │  ┌───┐       │  ┌───┐  │  ┌───┐  │  ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├┤ X ├──■──┤ X ├───────■──┤ X ├──■──┤ X ├──■──┤ H ├─╫──╫─┤M├
     ├───┤├───┤┌─┴─┐└───┘     ┌─┴─┐└───┘┌─┴─┐└───┘┌─┴─┐└───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├┤ X ├──────────┤ X ├─────┤ X ├─────┤ X ├──────╫──╫──╫─
     └───┘└───┘└───┘          └───┘     └───┘     └───┘      ║  ║  ║ 
c: 3/════════════════════════════════════════════════════════╩══╩══╩═
                                                             0  1  2 
Measurement: {'010': 1024}
Predicted type: balanced
